# Week 5 — Additional Models
**Internship:** IDX Exchange Data Science Program  
**Name:** Monika  
**Week:** 5  
**Dataset:** CRMLS Sold Properties, cleaned in Week 3

**Goal:** Try Decision Tree and Random Forest regressors, compare test R² against
the Week 4 Linear Regression baseline, and document how each model behaves
differently (overfitting, feature importance, non-linearity).

In [2]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_percentage_error

data_folder = r'C:\Users\monik\OneDrive - University of Illinois - Urbana\Desktop\IDX Exchange_DS\data\california'
model_df = pd.read_csv(data_folder + '\\cleaned_full.csv', parse_dates=['CloseDate_parsed'])

feature_cols = [
    'LivingArea', 'BedroomsTotal', 'BathroomsTotalInteger', 'LotSizeAcres',
    'PropertyAge', 'DaysOnMarket', 'Latitude', 'Longitude',
    'PoolPrivateYN', 'ViewYN', 'WaterfrontYN', 'BasementYN', 'AssociationFee',
    'LivingArea_missing', 'BathroomsTotalInteger_missing',
    'YearBuilt_missing', 'LotSizeAcres_missing', 'DaysOnMarket_anomaly'
]
target_col = 'ClosePrice'

def get_train_test_split(frame, test_month, window_months):
    frame = frame.copy()
    frame['YearMonth'] = frame['CloseDate_parsed'].dt.to_period('M')
    test_df = frame[frame['YearMonth'] == test_month]
    train_start = test_month - window_months
    train_df = frame[(frame['YearMonth'] >= train_start) & (frame['YearMonth'] < test_month)]
    return train_df.drop(columns='YearMonth'), test_df.drop(columns='YearMonth')

## 1. Re-establish the Baseline Split
Re-running the Week 4 window sweep with Linear Regression so this notebook picks
the same best window on its own (self-contained), then locking that window in
for all three models so the R² comparison is apples-to-apples.

In [3]:
test_month = pd.Period('2026-06', freq='M')
window_options = [3, 6, 9, 12, 18, 24]

sweep_results = []
for window in window_options:
    train_df, test_df = get_train_test_split(model_df, test_month, window)
    if len(train_df) < 50 or len(test_df) < 10:
        continue
    X_train, y_train = train_df[feature_cols], train_df[target_col]
    X_test, y_test = test_df[feature_cols], test_df[target_col]
    scaler = StandardScaler()
    lr = LinearRegression().fit(scaler.fit_transform(X_train), y_train)
    r2 = r2_score(y_test, lr.predict(scaler.transform(X_test)))
    sweep_results.append({'window_months': window, 'R2': r2})

sweep_df = pd.DataFrame(sweep_results)
BEST_WINDOW = int(sweep_df.loc[sweep_df['R2'].idxmax(), 'window_months'])
print(f'Locked-in window for comparison: {BEST_WINDOW} months')

train_df, test_df = get_train_test_split(model_df, test_month, BEST_WINDOW)
X_train, y_train = train_df[feature_cols], train_df[target_col]
X_test, y_test = test_df[feature_cols], test_df[target_col]

Locked-in window for comparison: 3 months


## 2. Baseline: Linear Regression
Same as Week 4, refit here just so it lives in the same comparison table below.

In [4]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

lr_model = LinearRegression().fit(X_train_scaled, y_train)
lr_train_r2 = r2_score(y_train, lr_model.predict(X_train_scaled))
lr_test_r2 = r2_score(y_test, lr_model.predict(X_test_scaled))
lr_test_mape = mean_absolute_percentage_error(y_test, lr_model.predict(X_test_scaled))

print(f'Linear Regression — Train R²: {lr_train_r2:.4f} | Test R²: {lr_test_r2:.4f}')

Linear Regression — Train R²: 0.4803 | Test R²: 0.4810


## 3. Decision Tree Regressor
Trees don't need scaled features, so I'm using the unscaled X_train/X_test here.
I'm tuning `max_depth` on a **held-out slice of the training data** (not the
test set) — tuning on test would make this comparison unfair to the baseline,
since I'd effectively be peeking at the answer.

In [5]:
X_tr, X_val, y_tr, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=42)

depth_options = [3, 5, 8, 10, 15, None]
depth_results = []
for depth in depth_options:
    dt = DecisionTreeRegressor(max_depth=depth, random_state=42).fit(X_tr, y_tr)
    val_r2 = r2_score(y_val, dt.predict(X_val))
    depth_results.append({'max_depth': depth, 'val_R2': val_r2})

depth_df = pd.DataFrame(depth_results)
print(depth_df)

best_depth = depth_df.loc[depth_df['val_R2'].idxmax(), 'max_depth']
print(f'Best max_depth: {best_depth}')

   max_depth    val_R2
0        3.0  0.416864
1        5.0  0.597126
2        8.0  0.745225
3       10.0  0.791687
4       15.0  0.785420
5        NaN  0.774802
Best max_depth: 10.0


In [6]:
best_depth = depth_df.loc[depth_df['val_R2'].idxmax(), 'max_depth']

if pd.notna(best_depth):
    best_depth = int(best_depth)

print(best_depth)

10


In [7]:
print(best_depth)
print(type(best_depth))

10
<class 'int'>


In [8]:
dt_model = DecisionTreeRegressor(max_depth=best_depth, random_state=42).fit(X_train, y_train)
dt_train_r2 = r2_score(y_train, dt_model.predict(X_train))
dt_test_r2 = r2_score(y_test, dt_model.predict(X_test))
dt_test_mape = mean_absolute_percentage_error(y_test, dt_model.predict(X_test))

print(f'Decision Tree — Train R²: {dt_train_r2:.4f} | Test R²: {dt_test_r2:.4f}')

Decision Tree — Train R²: 0.8577 | Test R²: 0.7846


## 4. Random Forest Regressor
Same idea, but averaging across many trees should reduce the overfitting a
single deep Decision Tree tends to show. Using a fixed n_estimators/max_depth
here rather than a full grid search, to keep this manageable — worth revisiting
with GridSearchCV later if this becomes the model I keep.

In [9]:
rf_model = RandomForestRegressor(
    n_estimators=300,
    max_depth=15,
    min_samples_leaf=5,
    random_state=42,
    n_jobs=-1
).fit(X_train, y_train)

rf_train_r2 = r2_score(y_train, rf_model.predict(X_train))
rf_test_r2 = r2_score(y_test, rf_model.predict(X_test))
rf_test_mape = mean_absolute_percentage_error(y_test, rf_model.predict(X_test))

print(f'Random Forest — Train R²: {rf_train_r2:.4f} | Test R²: {rf_test_r2:.4f}')

Random Forest — Train R²: 0.9398 | Test R²: 0.8731


## 5. Comparison Table

In [10]:
comparison_df = pd.DataFrame([
    {'model': 'Linear Regression', 'train_R2': lr_train_r2, 'test_R2': lr_test_r2, 'test_MAPE': lr_test_mape},
    {'model': 'Decision Tree', 'train_R2': dt_train_r2, 'test_R2': dt_test_r2, 'test_MAPE': dt_test_mape},
    {'model': 'Random Forest', 'train_R2': rf_train_r2, 'test_R2': rf_test_r2, 'test_MAPE': rf_test_mape},
])
comparison_df['overfit_gap'] = comparison_df['train_R2'] - comparison_df['test_R2']
comparison_df

,model,train_R2,test_R2,test_MAPE,overfit_gap
0,Linear Regression,0.480324,0.481007,0.464077,-0.000683
1,Decision Tree,0.857713,0.784613,0.211150,0.073100
2,Random Forest,0.939777,0.873122,0.151899,0.066656


In [11]:
importance_df = pd.DataFrame({
    'feature': feature_cols,
    'linear_coef_abs': np.abs(lr_model.coef_),
    'dt_importance': dt_model.feature_importances_,
    'rf_importance': rf_model.feature_importances_,
}).sort_values('rf_importance', ascending=False)
importance_df

,feature,linear_coef_abs,dt_importance,rf_importance
2,BathroomsTotalInteger,293799.260018,0.302125,0.271873
6,Latitude,43695.654770,0.246150,0.252690
7,Longitude,97012.342689,0.201216,0.211616
0,LivingArea,398918.268322,0.172816,0.178231
4,PropertyAge,245971.091719,0.048398,0.037674
3,LotSizeAcres,4531.849285,0.014459,0.022434
12,AssociationFee,28844.740518,0.008437,0.010339
5,DaysOnMarket,95744.384879,0.003621,0.009602
1,BedroomsTotal,61637.168473,0.001040,0.002516
8,PoolPrivateYN,22918.842390,0.000566,0.002004


## Summary of Findings — Week 5: Additional Models

**Setup:** Reused the 3-month training window (locked in from the Week 4 
Linear Regression sweep) with June 2026 as the test month, so all three 
models are compared on identical data splits.

**Decision Tree — Depth Tuning:**
Tested max_depth values of 3, 5, 8, 10, 15, and unlimited (None) using a 
train/validation split carved out of the training data (kept separate from 
the test set to avoid leaking test information into tuning). Validation R² 
rose steadily with depth up to 10 (0.417 → 0.597 → 0.745 → 0.792), then 
slightly declined at depth 15 (0.785) and unlimited depth (0.775) — a sign 
that deeper trees start overfitting the validation data itself rather than 
learning generalizable patterns. Best max_depth = 10.

**Results:**

| Model              | Train R² | Test R² | Overfit Gap | Test MAPE |
|--------------------|----------|---------|-------------|-----------|
| Linear Regression  | 0.480    | 0.481   | -0.001      | 0.464     |
| Decision Tree      | 0.858    | 0.785   | 0.073       | 0.211     |
| Random Forest      | 0.940    | 0.873   | 0.067       | 0.152     |

**Strengths/weaknesses observed:**
- **Linear Regression:** weakest model (test R² = 0.481, MAPE 46%), but 
  essentially zero overfit gap. Confirms the ceiling is about model capacity, 
  not generalization — a straight-line model can't capture nonlinear 
  interactions like location × size.
- **Decision Tree:** big jump to test R² = 0.785, but the largest overfit gap 
  of the three (0.073) — a single tree still memorizes more training-specific 
  patterns than an ensemble does, even after depth tuning.
- **Random Forest:** best performer (test R² = 0.873, MAPE 15.2%), and despite 
  being the most complex model, its overfit gap (0.067) is smaller than the 
  single Decision Tree's — averaging many trees cancels out individual trees' 
  overfitting tendencies.

**Feature importance:** Linear Regression weights LivingArea most heavily, 
while tree-based models rank BathroomsTotalInteger and Latitude/Longitude 
higher. This suggests location has a highly nonlinear relationship with price 
that tree models exploit through repeated splits but a linear model can't 
represent with just two coefficients — consistent with Linear Regression's 
much lower performance.

**Documented tradeoff:** The 3-month training window was chosen via the 
Linear Regression sweep and reused as-is for Decision Tree and Random Forest 
rather than re-optimizing per model — a reasonable simplification given time 
constraints.

**Best model overall: Random Forest (test R² = 0.873, MAPE = 15.2%)** — 
carried forward as the baseline to beat in Week 6 and beyond.